# Version 4 — Advanced RAG
**Project:** AI Placement Preparation Assistant using RAG

This notebook builds on V3 and adds **Advanced RAG** techniques.

### What is added in V4?

Basic RAG:
`Question → Vector Search → Relevant Chunks → Gemini → Answer`

Advanced RAG:
`Question → Query Rewriting → Hybrid Search → RRF → Reranking → Gemini → Answer + Sources`

| # | Stage | What we learn |
|---|---|---|
| 1 | Setup | Install required packages |
| 2 | API key | Secure Gemini key |
| 3 | PDF check | Confirm the 4 project PDFs exist |
| 4 | Document loading | Load all PDF pages |
| 5 | Metadata | Add technical / HR / resume / aptitude categories |
| 6 | Chunking | Split pages into smaller chunks |
| 7 | Embeddings | Convert chunks into vectors |
| 8 | FAISS | Semantic/vector search |
| 9 | BM25 | Keyword search |
| 10 | RRF | Combine vector + keyword results |
| 11 | Gemini | LLM used for rewriting, reranking and answering |
| 12 | Query rewriting | Turn one casual question into better search queries |
| 13 | Reranking | Re-order retrieved chunks by relevance |
| 14 | Complete Advanced RAG | Put all stages together |
| 15 | Testing | Supported + unsupported questions |
| 16 | Interactive chat | Ask your own questions |

### Important API-quota fix
The original notebook failed at query rewriting with a **429 `RESOURCE_EXHAUSTED`** error because the selected Gemini model had exhausted its free-tier quota.

This fixed version:
- keeps the Gemini model name in one variable;
- uses a lower-cost model by default;
- catches quota/rate-limit errors instead of crashing;
- falls back to the original question if rewriting fails;
- falls back to retrieval order if reranking fails;
- clearly reports when Gemini generation is unavailable.

**Important:** code cannot bypass an exhausted Gemini API quota. If the project/key has no remaining quota, wait for the reset or use a project/key with available quota.


## Step 1 — Setup
`rank_bm25` powers the keyword search.

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu rank_bm25 pypdf python-dotenv

## Step 2 — Gemini API key

In Google Colab, add a secret named **`Gemini_API`**:

**Colab → Secrets 🔑 → Add new secret → `Gemini_API`**

Never paste your real API key into a notebook cell or share it publicly.


In [ ]:
import os
from getpass import getpass

api_key = None

# 1) Google Colab: secret named Gemini_API  (left sidebar -> key icon)
try:
    from google.colab import userdata
    api_key = userdata.get("Gemini_API")
except Exception:
    pass

# 2) VS Code / local: a .env file in the project folder (GOOGLE_API_KEY=...) or an environment variable
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# 3) Last option: type it (the text is hidden)
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("No Gemini API key was provided.")

os.environ["GOOGLE_API_KEY"] = api_key
print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))


## Step 3 — Check the project PDFs

This project uses exactly these four placement-preparation PDFs:
- technical interview notes
- resume guidelines
- HR questions
- aptitude material

All four are required before continuing.


In [ ]:
import os

from pathlib import Path

# Works in Google Colab (PDFs uploaded to /content) AND in VS Code (project folder with data/).
if Path("/content/hr_questions.pdf").exists():
    DATA_DIR = "/content"
    WORK_DIR = "/content"                      # where saved indexes go in Colab
else:
    _here = Path.cwd()
    _project = next((p for p in [_here, _here.parent] if (p / "data" / "hr_questions.pdf").exists()), _here)
    DATA_DIR = str(_project / "data")
    WORK_DIR = str(_project / "vectorstores")  # saved indexes go here (ignored by git)
os.makedirs(WORK_DIR, exist_ok=True)
print("PDF folder :", DATA_DIR)

PDF_FILES = [
    f"{DATA_DIR}/technical_interview_.pdf",
    f"{DATA_DIR}/resume_guidelines.pdf",
    f"{DATA_DIR}/hr_questions.pdf",
    f"{DATA_DIR}/aptitude.pdf",
]

missing = [f for f in PDF_FILES if not os.path.exists(f)]

if missing:
    print("❌ Missing PDF files:")
    for f in missing:
        print("  ", f)
    raise FileNotFoundError(
        "Upload all 4 PDFs to Colab's Files panel, then run this cell again."
    )

print("✅ All 4 PDFs found.")


## Step 4 — Load PDFs
📍 **Roadmap 1**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

all_documents = []

for path in PDF_FILES:
    loader = PyPDFLoader(path)
    docs = loader.load()  # one LangChain Document per PDF page
    print(f"{os.path.basename(path):28s} -> {len(docs)} page(s)")
    all_documents.extend(docs)

print("\nTotal Documents (pages):", len(all_documents))


## Step 5 — Add category metadata

Metadata travels with every chunk.

Later we can use it to:
- filter searches by category;
- explain where an answer came from;
- display the source PDF and page number.


In [ ]:
CATEGORY_BY_FILE = {
    "technical_interview_.pdf": "technical",
    "resume_guidelines.pdf": "resume",
    "hr_questions.pdf": "hr",
    "aptitude.pdf": "aptitude",
}

for doc in all_documents:
    file_name = os.path.basename(doc.metadata["source"])
    doc.metadata["category"] = CATEGORY_BY_FILE.get(file_name, "other")

print("Example metadata:")
print(all_documents[0].metadata)


## Step 6 — Split into chunks
📍 **Roadmap 2**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(all_documents)

print("Pages :", len(all_documents))
print("Chunks:", len(chunks))
print("\n--- First chunk ---")
print(chunks[0].page_content[:1000])
print("\nMetadata:", chunks[0].metadata)


## Step 7 — Create embeddings

An embedding converts text into numbers (a vector).

We use the same embedding model for:
- document chunks
- user questions

This lets FAISS compare their vector representations.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

EMBEDDING_MODEL = "gemini-embedding-001"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL
)

try:
    vector = embeddings.embed_query("What is overfitting?")
    print("Embedding model:", EMBEDDING_MODEL)
    print("Vector length :", len(vector))
    print("First 5 values:", vector[:5])
except Exception as e:
    print("❌ Embedding request failed.")
    print(type(e).__name__, ":", e)
    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
        print("Gemini embedding quota is exhausted. Wait for quota reset or use a project/key with available quota.")
    raise


## Step 8 — Vector store (semantic search side)
📍 **Roadmap 4** — we reuse FAISS from V2.

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print("✅ FAISS ready.")
print("Vectors stored:", vectorstore.index.ntotal)


## Step 9 — Keyword search with BM25

We now create a second retrieval method.

- **FAISS:** finds text with similar meaning.
- **BM25:** is strong when important words/terms match exactly.

Advanced RAG combines both.


In [ ]:
from langchain_community.retrievers import BM25Retriever

import re


def bm25_tokens(text):
    """lower-case words without punctuation: 'method?' -> 'method'"""
    return re.findall(r"[a-z0-9]+", text.lower())


bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=bm25_tokens)
bm25_retriever.k = 6

q = "BM25 keyword search"

bm25_result = bm25_retriever.invoke(q)
vector_result = vector_retriever.invoke(q)

print("BM25 result  :", bm25_result[0].page_content[:150].replace("\n", " "))
print("Vector result:", vector_result[0].page_content[:150].replace("\n", " "))


## Step 10 — Reciprocal Rank Fusion (RRF)

FAISS and BM25 produce different ranking/score systems.

Instead of comparing their raw scores, **RRF combines their rankings**.

A document appearing near the top in both searches gets a stronger combined rank.


In [ ]:
def reciprocal_rank_fusion(result_lists, c=60):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (
                doc.metadata.get("source"),
                doc.metadata.get("page"),
                doc.page_content,
            )
            scores[key] = scores.get(key, 0) + 1 / (c + rank + 1)
            docs[key] = doc

    ordered_keys = sorted(scores, key=scores.get, reverse=True)
    return [docs[k] for k in ordered_keys]


def hybrid_search(query, top_n=8):
    vector_results = vector_retriever.invoke(query)
    keyword_results = bm25_retriever.invoke(query)

    return reciprocal_rank_fusion(
        [vector_results, keyword_results]
    )[:top_n]


test_q = "What is the STAR method?"
print("Hybrid results:")
for i, doc in enumerate(hybrid_search(test_q, top_n=4), start=1):
    print(
        f"{i}. {os.path.basename(doc.metadata['source'])} "
        f"- page {doc.metadata['page'] + 1}"
    )


## Step 11 — Gemini LLM

Gemini is used three times in V4: to **rewrite** the question, to **rerank** chunks and to **write the final answer**.

The model names are kept in one list. The free tier counts quota **per model**, so if one model is exhausted (429) the next one is tried automatically.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Keep the model names in one place. Free-tier quota is counted PER MODEL, so if the first
# model returns 429 the next one is tried automatically.
LLM_MODELS = ["gemini-3.5-flash-lite", "gemini-3.6-flash"]

llms = [
    ChatGoogleGenerativeAI(model=m, temperature=0, max_retries=1, timeout=60)
    for m in LLM_MODELS
]
llm = llms[0].with_fallbacks(llms[1:])

print("LLM models, in the order they are tried:", LLM_MODELS)


## Step 12 — Query rewriting

Students may ask:

> "how to talk about myself in interview"

A search system may work better with clearer variants such as:
- Tell me about yourself in an interview
- How should I introduce myself during an interview?
- What should I include in a self-introduction?

The original question is always kept as well.


In [ ]:
import re
import json
import time

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rewrite_prompt = ChatPromptTemplate.from_template(
    """Rewrite the student's question in up to 3 different ways to help search
placement-preparation notes (technical interview, aptitude, HR questions,
resume guidelines).

Use clear wording and likely keywords.
Keep the same meaning.
Output ONLY the rewritten questions, one per line, with no numbering.

Question: {question}"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def safe_llm_call(chain, inputs, fallback=None, label="Gemini call"):
    """Call Gemini. Retry when it is only busy (503). On any other failure return `fallback`."""
    for attempt in range(3):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            text = str(e)
            if ("503" in text or "UNAVAILABLE" in text) and attempt < 2:
                time.sleep(3 * (attempt + 1))
                continue
            print(f"⚠️ {label} failed: {type(e).__name__}: {text[:200]}")
            if "429" in text or "RESOURCE_EXHAUSTED" in text:
                print("Gemini quota/rate limit reached. Using the local fallback for this stage.")
            return fallback


def rewrite_query(question):
    text = safe_llm_call(
        rewrite_chain,
        {"question": question},
        fallback=None,
        label="Query rewriting",
    )

    if text is None:                      # Gemini failed -> search with the original question only
        return [question]

    rewrites = []
    for line in text.splitlines():
        cleaned = re.sub(r"^[\s\-•*\d\.)]+", "", line).strip()
        if cleaned and cleaned.lower() != question.lower():
            rewrites.append(cleaned)

    return [question] + rewrites[:3]


print(rewrite_query("how to talk about myself in interview"))


### What to observe

The result contains the **original question plus rewritten versions**.

If Gemini quota is unavailable, the notebook safely falls back to the original question instead of stopping with an error.


## Step 13 — Reranking

Retrieval is a quick first filter.

A **reranker** examines the retrieved chunks more carefully and gives each one a relevance score from 0–10.

Then we keep the highest-scoring chunks for the final answer.


In [ ]:
rerank_prompt = ChatPromptTemplate.from_template(
    """You are a strict relevance judge.

Question: {question}

Below are numbered text chunks. Give each chunk a relevance score from 0 to 10.
10 = directly answers the question.
0 = unrelated.

Return ONLY a JSON list of numbers, one score per chunk, in order.
Example: [8, 2, 0, 9]

Chunks:
{chunks}"""
)

rerank_chain = rerank_prompt | llm | StrOutputParser()


def rerank(question, docs, top_n=4):
    if not docs:
        return []

    numbered = "\n\n".join(
        f"[{i}] {doc.page_content}" for i, doc in enumerate(docs)
    )

    raw = safe_llm_call(
        rerank_chain,
        {"question": question, "chunks": numbered},
        fallback=None,
        label="Reranking"
    )

    if raw is None:
        # If Gemini is unavailable, preserve retrieval order.
        return [(doc, 5.0) for doc in docs[:top_n]]

    try:
        match = re.search(r"\[[\s\S]*?\]", raw)
        scores = json.loads(match.group()) if match else None

        if not isinstance(scores, list) or len(scores) != len(docs):
            raise ValueError("Invalid score list.")

        scores = [float(s) for s in scores]
    except Exception:
        print("⚠️ Could not read reranking scores. Keeping retrieval order.")
        scores = [5.0] * len(docs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda item: item[1],
        reverse=True
    )

    return ranked[:top_n]


test_q = "What is the STAR method?"
test_docs = hybrid_search(test_q, top_n=8)

for doc, score in rerank(test_q, test_docs):
    print(
        f"score={score:.1f} | "
        f"{os.path.basename(doc.metadata['source'])} "
        f"- page {doc.metadata['page'] + 1}"
    )


## Step 14 — Complete Advanced RAG

Final flow:

`Question`
→ **Query rewriting**
→ **Hybrid retrieval (FAISS + BM25)**
→ **RRF**
→ **Reranking**
→ **Relevance check**
→ **Gemini answer**
→ **Source + page**

The answer is instructed to use **only retrieved document context**.


In [ ]:
NOT_FOUND = "I could not find the answer in the uploaded documents."

answer_prompt = ChatPromptTemplate.from_template(
    """You are an AI Placement Preparation Assistant.

Answer the question using ONLY the context below.

If the answer is not supported by the context, reply exactly:
"{not_found}"

Do not use outside knowledge.
Keep the answer clear and beginner-friendly.

Context:
{context}

Question: {question}

Answer:"""
)

answer_chain = answer_prompt | llm | StrOutputParser()


In [ ]:
def format_context(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def format_sources(docs):
    seen = set()
    lines = []

    for doc in docs:
        name = os.path.basename(doc.metadata.get("source", "unknown"))
        page = int(doc.metadata.get("page", 0)) + 1
        key = (name, page)

        if key not in seen:
            seen.add(key)
            lines.append(f"{name} - page {page}")

    return lines


def answer_question(question, docs):
    context = format_context(docs)

    return safe_llm_call(
        answer_chain,
        {
            "context": context,
            "question": question,
            "not_found": NOT_FOUND,
        },
        fallback=None,
        label="Final answer generation"
    )


MIN_RELEVANCE = 5.0


def advanced_ask(question, verbose=True):
    # 1. Query rewriting
    queries = rewrite_query(question)

    if verbose:
        print("Queries used:")
        for q in queries:
            print(" -", q)
        print()

    # 2. Hybrid retrieval for every query
    all_lists = []

    for q in queries:
        all_lists.append(vector_retriever.invoke(q))
        all_lists.append(bm25_retriever.invoke(q))

    # 3. Merge rankings with RRF
    candidates = reciprocal_rank_fusion(all_lists)[:12]   # task document: top 10-20

    if not candidates:
        print("\nAnswer:\n", NOT_FOUND)
        return

    # 4. Rerank
    ranked = rerank(question, candidates, top_n=4)

    if verbose:
        print("Reranked chunks:")
        for doc, score in ranked:
            print(
                f"  score={score:.1f} | "
                f"{os.path.basename(doc.metadata['source'])} "
                f"- page {doc.metadata['page'] + 1}"
            )

    # 5. Relevance gate
    if not ranked or ranked[0][1] < MIN_RELEVANCE:
        print("\nAnswer:\n", NOT_FOUND)
        return

    docs = [doc for doc, score in ranked if score >= MIN_RELEVANCE]

    # 6. Final generation
    answer = answer_question(question, docs)

    if answer is None:
        print("\nAnswer could not be generated because Gemini is unavailable.")
        return

    print("\nAnswer:\n", answer)

    if NOT_FOUND not in answer:
        print("\nSources:")
        for source in format_sources(docs):
            print(" -", source)


## Step 15 — Test the complete pipeline

One question that is covered by the PDFs, then (optionally) more. Each question uses about 3 Gemini calls (rewrite + rerank + answer), so run the optional ones one at a time.

In [ ]:
# Main test
advanced_ask("how to talk about myself in interview")

# Optional tests — run them one at a time if Gemini quota is available.
#
# advanced_ask("difference between bagging and boosting")
# advanced_ask("how to find average speed when distance is same")
# advanced_ask("Who won the cricket world cup?")


### Expected behavior

- A question covered by the PDFs should produce an answer and source pages.
- A question outside the uploaded documents should be refused with:
  **`I could not find the answer in the uploaded documents.`**
- If Gemini quota is exhausted, the notebook should show a clear quota message instead of crashing.


## Step 16 — Interactive chat

Run the next cell when you want to ask your own questions.

Type `exit` or `quit` to stop.

**Tip:** interactive chat makes several Gemini calls per question because V4 performs rewriting + reranking + final generation. Use it only when your API quota is available.


In [ ]:
print("Interactive Advanced RAG chat started.")
print("Type 'exit' or 'quit' to stop.")

while True:
    q = input("\nYou: ").strip()

    if q.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    if not q:
        continue

    advanced_ask(q, verbose=False)


# 🎓 Viva Corner

### Q1. Why is V4 called Advanced RAG?
Because it improves the basic RAG pipeline with **query rewriting, hybrid retrieval, RRF and reranking**.

### Q2. Why use both FAISS and BM25?
FAISS is good at **semantic similarity**; BM25 is good at **keyword/exact-term matching**.

### Q3. Why use RRF?
FAISS and BM25 use different scoring systems. RRF combines their **rank positions** instead of comparing incompatible raw scores.

### Q4. What is query rewriting?
It creates clearer search versions of the user's question while keeping the same meaning.

### Q5. What is reranking?
It takes the initially retrieved chunks and orders them again according to how relevant they are to the original question.

### Q6. What happens if Gemini quota is exhausted?
The notebook catches the API error and uses fallbacks where possible. Final answer generation cannot happen without an available LLM quota.

### Q7. What is the complete V4 flow?
**Question → Query Rewriting → FAISS + BM25 → RRF → Reranking → Relevance Check → Gemini → Answer + Sources.**
